In [ ]:
%%capture
# !uv pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl
# !aria2c -x 16 -s 16 -d ../models -o fasttext_corpus.zip https://vectors.nlpl.eu/repository/20/24.zip
# !unzip ../models/fasttext_corpus.zip -d ../models/fasttext

## импорты

In [ ]:
import re
import os
from tqdm.auto import tqdm

import numpy as np
import pandas as pd
tqdm.pandas()
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True) 
import matplotlib.pyplot as plt

import torch
import torchaudio.transforms as T
from sklearn.metrics import f1_score

import librosa

# tmux attach -t download_torch

from num2words import num2words
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

custom_exceptions = {
    'not', 'no', 'never', 'nor', 'neither', 
    'but', 'however', 'although', 'very', 
    'too', 'extremely', 'again', 'still', 
    'only', 'just', 'against', 'up', 'down'
}
stop_words = set(stopwords.words('english')) - custom_exceptions

import spacy
nlp = spacy.load("en_core_web_sm")

import seaborn
seaborn.set_theme(palette='summer')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
num_workers = os.cpu_count() if device == "cpu" else 0
batch_size = 32
sample_rate = 16_000
loader_args = {
    'batch_size': batch_size,
    'num_workers': num_workers
}
device

INFO: Pandarallel will run on 4 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


[nltk_data] Downloading package stopwords to /home/ext-
[nltk_data]     yankin@ad.speechpro.com/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/ext-
[nltk_data]     yankin@ad.speechpro.com/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/ext-
[nltk_data]     yankin@ad.speechpro.com/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


'cuda'

In [60]:
models_path = '../models'
features_path = '../features'

In [61]:
train_df = pd.read_csv('../datasets/train.csv')
val_df = pd.read_csv('../datasets/val.csv')
test_df = pd.read_csv('../datasets/test.csv')

In [62]:
target_cols = ['happy', 'sad', 'anger', 'surprise', 'disgust', 'fear']

train_labels = (train_df[target_cols].values > 0).astype(int)
val_labels = (val_df[target_cols].values > 0).astype(int)
test_labels = (test_df[target_cols].values > 0).astype(int)

## подготовка датасета текстовой модальности
- приведение к нижнему регистру, нормализация чисел, лемматизация, фильтрация: по англ. алфавиту, стоп-словам и len(слово) > 2;
- `вход -> выход`: `(train_df, val_df, test_df : DataFrame) -> (train_text_loader, val_text_loader, test_text_loader : DataLoader)`

### общая обработка

In [4]:
RE_SUFFIX = re.compile(r'\b(\d+)-[a-z]+\b')
RE_MIXED = re.compile(r'\b\d+[a-z]+\b')
RE_NUMBER = re.compile(r'\b\d+(?:\.\d+)?')

def text2num(match: re.Match) -> str:
    val = match.group()
    if len(val) > 15: 
        return val
    try:
        return num2words(val, lang='en')
    except (ValueError, TypeError):
        return val

def split_mixed_numchar(match: re.Match) -> str:
    word = match.group()
    num = "".join(c for c in word if c.isdigit())
    txt = "".join(c for c in word if c.isalpha())
    
    result = f"{num2words(num, lang='en')} {txt}".strip()

    return result

def prepare(text: str) -> List[str]:
    text = text.lower()
    text = RE_SUFFIX.sub(r'\1', text)
    text = RE_MIXED.sub(split_mixed_numchar, text)
    text = RE_NUMBER.sub(text2num, text)
    
    doc = nlp(text)
    tokens = [
        t.lemma_.lower() for t in doc 
        if t.is_alpha and len(t) > 1 and 
        (t.text.lower() not in stop_words or t.text.lower() in custom_exceptions)
    ]
    return tokens

train_text = train_df['text'].astype(str).parallel_apply(prepare)
val_text = val_df['text'].astype(str).parallel_apply(prepare)
test_text = test_df['text'].astype(str).parallel_apply(prepare)
train_text.head(3)

0    [key, part, people, use, solve, issue, whether...
1    [able, find, solution, least, bring, answer, t...
2    [key, polymer, bring, technical, aspect, opera...
Name: text, dtype: object

### выделение признаков Fasttext

In [15]:
import fasttext

train_sentences = train_text.tolist()
with open('train_data.txt', 'w', encoding='utf-8') as f:
    for tokens in train_sentences:
        f.write(" ".join(tokens) + "\n")

model_finetuned = fasttext.train_unsupervised(
    input="train_data.txt",
    model='skipgram',
    pretrainedVectors='../models/fasttext/model.bin',
    epoch=5,
    lr=0.025,
    dim=300
)

model_finetuned.save_model('../models/fasttext_finetuned.bin')

Read 0M words
Number of words:  3599
Number of labels: 0
Progress: 100.0% words/sec/thread:   55234 lr:  0.000000 avg.loss:  2.533199 ETA:   0h 0m 0s


In [20]:
def get_avg_vector(tokens, model):
    vectors = [model.get_word_vector(t) for t in tokens]
    if not vectors:
        return np.zeros(model.get_dimension())
    return np.mean(vectors, axis=0)

fasttext_features = train_text.progress_apply(
    lambda text_tokens: get_avg_vector(text_tokens, model_finetuned)
)

df_train_ft = pd.DataFrame(fasttext_features.tolist())
df_train_ft.to_csv(f'{features_path}/ft_train_features.csv', index=False)

print(f"Размерность fasttext признаков: Train - {df_train_ft.shape}")

  0%|          | 0/16274 [00:00<?, ?it/s]

Размерность fasttext признаков: Train - (16274, 300)


In [21]:
fasttext_val_features = val_text.progress_apply(
    lambda text_tokens: get_avg_vector(text_tokens, model_finetuned)
)
df_val_ft = pd.DataFrame(fasttext_val_features.tolist())
df_val_ft.to_csv(f'{features_path}/ft_val_features.csv', index=False)

fasttext_test_features = test_text.progress_apply(
    lambda text_tokens: get_avg_vector(text_tokens, model_finetuned)
)
df_test_ft = pd.DataFrame(fasttext_test_features.tolist())
df_test_ft.to_csv(f'{features_path}/ft_test_features.csv', index=False)

print(f"Размерности fasttext признаков: Val - {df_val_ft.shape}, Test - {df_test_ft.shape}")

  0%|          | 0/1861 [00:00<?, ?it/s]

  0%|          | 0/4662 [00:00<?, ?it/s]

Размерности fasttext признаков: Val - (1861, 300), Test - (4662, 300)


### выделение признаков TF-IDF

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib


train_str = train_text.progress_apply(lambda x: " ".join(x))
val_str = val_text.progress_apply(lambda x: " ".join(x))
test_str = test_text.progress_apply(lambda x: " ".join(x))

tfidf_vectorizer = TfidfVectorizer(
    max_features=1000,
    ngram_range=(1, 2),
    min_df=3
)

tfidf_train = tfidf_vectorizer.fit_transform(train_str).toarray()

tfidf_val = tfidf_vectorizer.transform(val_str).toarray()
tfidf_test = tfidf_vectorizer.transform(test_str).toarray()

joblib.dump(tfidf_vectorizer, f'{models_path}/tfidf_vectorizer.pkl')

pd.DataFrame(tfidf_train).to_csv(f'{features_path}/tfidf_train_features.csv', index=False)
pd.DataFrame(tfidf_val).to_csv(f'{features_path}/tfidf_val_features.csv', index=False)
pd.DataFrame(tfidf_test).to_csv(f'{features_path}/tfidf_test_features.csv', index=False)

print(f"Размерность TF-IDF признаков: {tfidf_train.shape[1]}")

  0%|          | 0/16274 [00:00<?, ?it/s]

  0%|          | 0/1861 [00:00<?, ?it/s]

  0%|          | 0/4662 [00:00<?, ?it/s]

Размерность TF-IDF признаков: 1000


## подготовка датасета аудио модальности
- извлечение фрагмента по времени (start_time, end_time), ресемплирование до ?? кГц, преобразование в моно, приведение к фиксированной длине (padding/cropping) и опциональная аугментация;
- `вход -> выход`: `(train_df, val_df, test_df : DataFrame) -> (train_audio_loader, val_audio_loader, test_audio_loader : DataLoader)`

### общая обработка и извлечение mfcc и mel признаков 

In [5]:
n_fft = 1024
hop_length = 512

n_mfcc = 20
n_mels = 128

In [12]:
def process_audio_full(row, n_mfcc=20, n_mels=128):
    file_path = f"../datasets/audio/{row['video']}.wav"
    start = row['start_time']
    dur = row['end_time'] - start
    
    y, sr = librosa.load(file_path, sr=22050, offset=start, duration=dur, res_type='kaiser_fast')
    y, _ = librosa.effects.trim(y)
    
    if len(y) < 1024: return None 
    
    y = y / (np.max(np.abs(y)) + 1e-9)
    y = np.append(y[0], y[1:] - 0.97 * y[:-1])
    
    # MFCC (20 mean + 20 std = 40)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    mfcc_feat = np.hstack([np.mean(mfcc, axis=1), np.std(mfcc, axis=1)])
    
    # Mel (128 mean + 128 std = 256)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_feat = np.hstack([np.mean(mel_db, axis=1), np.std(mel_db, axis=1)])
    
    return {'mfcc': mfcc_feat.astype('float32'), 'mel': mel_feat.astype('float32')}

In [ ]:
def save_audio_features(df, split_name):
    results = df.parallel_apply(process_audio_full, axis=1)
    res_df = results.apply(pd.Series)

    for feat in ['mfcc', 'mel']:
        dim = 40 if feat == 'mfcc' else 256
        data = np.stack(res_df[feat].apply(lambda x: x if isinstance(x, np.ndarray) else np.zeros(dim)))
        pd.DataFrame(data).to_csv(f'{features_path}/audio_{feat}_{split_name}.csv', index=False)

for df, name in zip([train_df, val_df, test_df], ['train', 'val', 'test']):
    save_audio_features(df, name)

In [ ]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

def scale_and_save_features(feature_types):
    features_path = '../features'
    for feat in feature_types:
        train_path = f'{features_path}/{feat}_train.csv'
        val_path = f'{features_path}/{feat}_val.csv'
        test_path = f'{features_path}/{feat}_test.csv'
        
        df_train = pd.read_csv(train_path)
        df_val = pd.read_csv(val_path)
        df_test = pd.read_csv(test_path)
        
        scaler = StandardScaler()
        scaled_train = scaler.fit_transform(df_train)
        
        scaled_val = scaler.transform(df_val)
        scaled_test = scaler.transform(df_test)
        
        pd.DataFrame(scaled_train).to_csv(f'{features_path}/scaled_{feat}_train.csv', index=False)
        pd.DataFrame(scaled_val).to_csv(f'{features_path}/scaled_{feat}_val.csv', index=False)
        pd.DataFrame(scaled_test).to_csv(f'{features_path}/scaled_{feat}_test.csv', index=False)

scale_and_save_features(['mfcc', 'mel'])